##### Load config

In [2]:
%run 00_config

StatementMeta(, 1377704e-5b35-418d-b3bb-eab0cfa41436, 3, Finished, Available, Finished, True)

FHIR Pipeline Config loaded
  FHIR URL   : https://hapi.fhir.org/baseR4
  Resources  : ['Patient', 'Encounter', 'Observation', 'Condition']
  Ingest days: 3
  Schemas    : bronze | silver | gold
  Naming     : tables → tbl_  | views → vw_

Table names:
  bronze.tbl_patient                  → silver.tbl_patient                  → gold.vw_dim_patient
  bronze.tbl_encounter                → silver.tbl_encounter                → gold.vw_fact_encounter
  bronze.tbl_observation              → silver.tbl_observation              → gold.vw_fact_observation
  bronze.tbl_condition                → silver.tbl_condition                → gold.vw_fact_condition


##### Imports

In [3]:
import requests
import json
from datetime import datetime, timedelta
from notebookutils import mssparkutils
from pyspark.sql import functions as F
from pyspark.sql.types import *


StatementMeta(, 1377704e-5b35-418d-b3bb-eab0cfa41436, 4, Finished, Available, Finished, False)

##### Functions

In [4]:
def fetch_resource_pages(resource: str, date_str: str) -> list:
    """
    Fetch all paginated FHIR records for a resource on a given date.
    Uses CONFIG for page_size, page_cap, timeout.
    """
    records  = []
    url      = f"{CONFIG['fhir_base_url']}/{resource}"
    params   = {
        "_count"      : CONFIG["page_size"],
        "_lastUpdated": f"ge{date_str}",
        "_format"     : "json"
    }
    page_num = 0

    while url:
        resp = requests.get(
            url,
            params=params if page_num == 0 else None,
            timeout=CONFIG["timeout_seconds"]
        )
        resp.raise_for_status()
        bundle  = resp.json()
        entries = bundle.get("entry", [])
        records.extend(entries)
        print(f"    Page {page_num + 1}: {len(entries)} records")

        # Follow next page link
        url = None
        for link in bundle.get("link", []):
            if link["relation"] == "next":
                url = link["url"]
                break

        page_num += 1
        if page_num >= CONFIG["page_cap"]:
            print(f"    Page cap ({CONFIG['page_cap']}) reached.")
            break

    return records

print("Fetch function ready")

StatementMeta(, 1377704e-5b35-418d-b3bb-eab0cfa41436, 5, Finished, Available, Finished, False)

Fetch function ready


In [5]:
BRONZE_SCHEMA = StructType([
    StructField("resource_type",        StringType(), True),
    StructField("resource_id",          StringType(), True),
    StructField("raw_json",             StringType(), True),
    StructField("extraction_timestamp", StringType(), True),
    StructField("api_url",              StringType(), True),
    StructField("load_date",            StringType(), True),
])

print("Bronze schema defined")

StatementMeta(, 1377704e-5b35-418d-b3bb-eab0cfa41436, 6, Finished, Available, Finished, False)

Bronze schema defined


In [6]:
extraction_ts = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S")

for resource in CONFIG["resources"]:
    print(f"\n{'='*50}")
    print(f"Resource: {resource}")
    print(f"Target  : {bronze_table(resource)}")

    for day_offset in range(CONFIG["ingest_days"]):
        date_str = (datetime.utcnow() - timedelta(days=day_offset)).strftime("%Y-%m-%d")
        print(f"\n  Date: {date_str}")

        try:
            records = fetch_resource_pages(resource, date_str)
        except Exception as e:
            print(f"  ERROR fetching {resource} on {date_str}: {e}")
            continue

        if not records:
            print("  No records found, skipping.")
            continue

        # ── 1. Save raw JSON to Files layer ──────────────
        file_path = raw_path(resource, date_str)
        mssparkutils.fs.put(
            file_path,
            json.dumps(records, indent=2),
            overwrite=True
        )
        print(f"  Raw JSON saved → {file_path}")

        # ── 2. Build bronze rows ──────────────────────────
        rows = []
        for entry in records:
            res_obj = entry.get("resource", entry)
            rows.append((
                resource,
                res_obj.get("id", "unknown"),
                json.dumps(res_obj),
                extraction_ts,
                f"{CONFIG['fhir_base_url']}/{resource}",
                date_str
            ))

        df = spark.createDataFrame(rows, BRONZE_SCHEMA)

        # ── 3. Append to bronze Delta table ──────────────
        tbl = bronze_table(resource)
        df.write.format("delta") \
          .mode("append") \
          .partitionBy("load_date") \
          .option("mergeSchema", "true") \
          .saveAsTable(tbl)

        print(f"  Appended {len(rows)} rows → {tbl}")

print("\nBronze ingestion complete!")

StatementMeta(, 1377704e-5b35-418d-b3bb-eab0cfa41436, 7, Finished, Available, Finished, False)


Resource: Patient
Target  : bronze.tbl_patient

  Date: 2026-06-16
    Page 1: 50 records
    Page 2: 50 records
    Page 3: 50 records
    Page 4: 50 records
    Page 5: 50 records
    Page cap (5) reached.
  Raw JSON saved → Files/raw/patient/2026-06-16/response.json
  Appended 250 rows → bronze.tbl_patient

  Date: 2026-06-15
    Page 1: 50 records
    Page 2: 50 records
    Page 3: 50 records
    Page 4: 50 records
    Page 5: 50 records
    Page cap (5) reached.
  Raw JSON saved → Files/raw/patient/2026-06-15/response.json
  Appended 250 rows → bronze.tbl_patient

  Date: 2026-06-14
    Page 1: 50 records
    Page 2: 50 records
    Page 3: 50 records
    Page 4: 50 records
    Page 5: 50 records
    Page cap (5) reached.
  Raw JSON saved → Files/raw/patient/2026-06-14/response.json
  Appended 250 rows → bronze.tbl_patient

Resource: Encounter
Target  : bronze.tbl_encounter

  Date: 2026-06-16
    Page 1: 50 records
    Page 2: 50 records
    Page 3: 50 records
    Page 4: 12 reco